# ЛР3 — практика A: самостоятельная работа от `.fit()` к одному training step

**Тема:** `Tensor`, `DataLoader`, `nn.Module`, logits, loss, gradients и optimizer  
**Формат:** компактная самостоятельная часть после первой практики ЛР3  
**Контроль:** практика B продолжит тот же сюжет полным training loop; защита A+B общая.

Центральный вопрос:

> **Какие операции скрывает высокоуровневый `.fit()`, когда нейросеть делает один шаг обучения?**

## 0. Окружение и данные

Используем подготовленные массивы Titanic из ЛР2-A. Если артефакт ещё не создан, сначала завершите `02_tabular_data/A_titanic/student.ipynb`.

In [ ]:
SEED = 42
from pathlib import Path
import random, os
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score, f1_score
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
DEVICE = torch.device("cpu")

artifact_candidates = [
    Path("../../02_tabular_data/A_titanic/artifacts/lr2_titanic_prepared.npz"),
    Path("../../../shared/02_tabular_data/A_titanic/artifacts/lr2_titanic_prepared.npz"),
    Path("artifacts/lr2_titanic_prepared.npz"),
]
ARTIFACT_PATH = next((p for p in artifact_candidates if p.exists()), None)
if ARTIFACT_PATH is None:
    raise FileNotFoundError("Не найден артефакт ЛР2-A: lr2_titanic_prepared.npz")

with np.load(ARTIFACT_PATH) as data:
    X_train = data["X_train"].astype(np.float32)
    X_val = data["X_val"].astype(np.float32)
    y_train = data["y_train"].astype(np.int64)
    y_val = data["y_val"].astype(np.int64)

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)

# 1. Задание 1 — high-level baseline

Обучите `MLPClassifier(hidden_layer_sizes=(16,), random_state=SEED, max_iter=500)` и получите accuracy/F1 на validation.

После запуска ответьте: **какие части процесса обучения здесь спрятаны внутри `.fit()`?** Назовите не меньше четырёх.

In [ ]:
# TODO: sklearn_mlp = ...
# TODO: fit, predict, accuracy, F1.


# 2. Задание 2 — TensorDataset и DataLoader

1. Преобразуйте `X_train` в `float32` tensor.
2. Для `BCEWithLogitsLoss` преобразуйте target в `float32`.
3. Создайте `TensorDataset` и `DataLoader(batch_size=32, shuffle=True)` с фиксированным generator seed.
4. Возьмите один batch и объясните его формы `(B, D)` и `(B,)`.

In [ ]:
BATCH_SIZE = 32
# TODO: X_train_t, y_train_t, dataset, loader, xb, yb.


# 3. Задание 3 — описать MLP через `nn.Module`

Архитектура:

```text
input_dim → Linear(16) → ReLU → Linear(1) → one logit
```

Последний слой **не содержит Sigmoid**: используем `BCEWithLogitsLoss`. `forward()` должен вернуть форму `(batch,)`.

In [ ]:
class TitanicMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim=16):
        super().__init__()
        # TODO

    def forward(self, x):
        # TODO
        raise NotImplementedError


# 4. Задание 4 — smoke test до обучения

На одном batch:
1. создайте модель;
2. получите logits;
3. вычислите `BCEWithLogitsLoss`;
4. проверьте `logits.shape == yb.shape` и конечность loss.

Почему такой тест полезнее, чем сразу запускать 200 эпох?

In [ ]:
# TODO: model, criterion, logits, loss, asserts.


# 5. Задание 5 — один training step

Выполните ровно один шаг:

```python
optimizer.zero_grad()
logits = model(xb)
loss = criterion(logits, yb)
loss.backward()
optimizer.step()
```

Дополнительно сохраните копию весов первого `Linear` **до** шага и докажите, что после `optimizer.step()` хотя бы один параметр изменился.

In [ ]:
# TODO: optimizer и один training step + проверка изменения параметров.


# 6. Мини-checkpoint и вывод

Ответьте коротко:

1. Что делает `backward()`, а что делает `optimizer.step()`?
2. Почему `zero_grad()` вызывают перед новым шагом?
3. Почему logits и probability — не одно и то же?
4. Что означает `B` и `D` в batch форме `(B, D)`?
5. Какие пять действий вы теперь можете назвать внутри одного training step?

**Дальше:** в `B_training_and_regularization/student.ipynb` эти шаги будут собраны в эпохи, history и controlled experiment с Dropout.